# LLM-Based Reasoning for FailureSensorIQ

This notebook is written for GitHub upload.

It is intentionally simple and heavily commented.

No Hugging Face login, token, Kaggle secret, or API key is used.

The notebook evaluates:

1. Zero-shot Qwen2.5-7B-Instruct
2. Few-shot Qwen2.5-7B-Instruct
3. QLoRA fine-tuned Qwen2.5-7B-Instruct

The notebook also computes:

- main test metrics,
- error analysis,
- disagreement analysis,
- robustness metrics.

In [ ]:
# If you run this notebook on Kaggle, uncomment the next line if packages are missing.
# !pip install -q transformers accelerate bitsandbytes peft trl datasets openpyxl

In [ ]:
# =========================
# 1. Import packages
# =========================

import os
import re
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [ ]:
# =========================
# 2. Configuration
# =========================

RANDOM_SEED = 42
TEST_SIZE = 0.20

# Model used in the paper.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# These switches make the notebook easy to test.
RUN_ZERO_SHOT = True
RUN_FEW_SHOT = True
RUN_QLORA = True

# Set these to small numbers for a quick test.
# Set them to None for final full experiments.
MAX_EVAL_SAMPLES = None
MAX_TRAIN_SAMPLES = None

# Few-shot examples used in prompt.
N_FEW_SHOT_EXAMPLES = 4

# Text generation settings.
MAX_NEW_TOKENS = 8
GEN_BATCH_SIZE = 4
MAX_SEQ_LENGTH = 768

# QLoRA settings.
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 0.3

# Output folder.
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_OUTPUT_DIR = Path("../models/qwen25_qlora_adapter")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 3. Helper functions for files
# =========================

def find_file(file_name):
    """Find a file in common local and Kaggle folders."""
    search_roots = [
        Path("../data"),
        Path("data"),
        Path("/kaggle/input"),
        Path("/mnt/data"),
        Path("."),
    ]

    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.rglob(file_name))
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not find {file_name}. Put it in data/ or Kaggle input.")


def load_jsonl(path):
    """Load a JSONL file into a pandas DataFrame."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def get_correct_index(correct_list):
    """Return the index of the correct option."""
    for i, value in enumerate(correct_list):
        if bool(value):
            return i
    return None


def normalize_dataframe(df):
    """Add common columns needed for evaluation."""
    df = df.copy()
    df["id"] = df["id"].astype(int)
    df["correct_index"] = df["correct"].apply(get_correct_index)
    df["correct_option"] = df.apply(lambda r: r["options"][r["correct_index"]], axis=1)
    df["correct_label"] = df.apply(lambda r: r["option_ids"][r["correct_index"]], axis=1)
    df["n_options"] = df["options"].apply(len)

    for col in ["asset_name", "question_type", "relevancy"]:
        if col not in df.columns:
            df[col] = "unknown"
        df[col] = df[col].fillna("unknown")

    return df

In [ ]:
# =========================
# 4. Load datasets
# =========================

main_path = find_file("all.jsonl")
simple_path = find_file("all_10_options_all_perturbed_simple.jsonl")
complex_path = find_file("all_10_options_perturbed_complex.jsonl")

main_df = normalize_dataframe(load_jsonl(main_path))
simple_df = normalize_dataframe(load_jsonl(simple_path))
complex_df = normalize_dataframe(load_jsonl(complex_path))

print("Main data:", main_df.shape)
print("Simple perturbed data:", simple_df.shape)
print("Complex perturbed data:", complex_df.shape)

In [ ]:
# =========================
# 5. Train/test split
# =========================

train_df, test_df = train_test_split(
    main_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Optional smaller run for debugging.
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.head(MAX_TRAIN_SAMPLES).copy()

if MAX_EVAL_SAMPLES is not None:
    test_df = test_df.head(MAX_EVAL_SAMPLES).copy()

# Select matching perturbed rows using test ids.
test_ids = set(test_df["id"].tolist())
simple_test_df = simple_df[simple_df["id"].isin(test_ids)].copy().reset_index(drop=True)
complex_test_df = complex_df[complex_df["id"].isin(test_ids)].copy().reset_index(drop=True)

# Add original metadata to perturbed rows.
metadata_cols = ["asset_name", "question_type", "relevancy"]
id_to_meta = main_df.set_index("id")[metadata_cols].to_dict("index")
for df in [simple_test_df, complex_test_df]:
    for col in metadata_cols:
        df[col] = df.apply(lambda r: id_to_meta.get(r["id"], {}).get(col, r[col]), axis=1)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Simple perturbed test size:", len(simple_test_df))
print("Complex perturbed test size:", len(complex_test_df))

In [ ]:
# =========================
# 6. Prompt builders
# =========================

def format_options(row):
    """Convert options into a readable string."""
    lines = []
    for label, option in zip(row["option_ids"], row["options"]):
        lines.append(f"{label}. {option}")
    return "
".join(lines)


def zero_shot_messages(row):
    """Prompt with no examples."""
    user_text = f"""
You are an industrial asset health monitoring assistant.
Read the question and select the correct answer option.
Return only one option label.

Question:
{row['question']}

Options:
{format_options(row)}
""".strip()

    return [
        {"role": "system", "content": "You answer industrial multiple-choice questions."},
        {"role": "user", "content": user_text},
    ]


def few_shot_messages(row, examples):
    """Prompt with a few solved examples."""
    example_blocks = []
    for i, ex in enumerate(examples, start=1):
        block = f"""
Example {i}:
Question:
{ex['question']}

Options:
{format_options(ex)}

Answer: {ex['correct_label']}
""".strip()
        example_blocks.append(block)

    user_text = f"""
You are an industrial asset health monitoring assistant.
Select the correct answer option.
Return only one option label.

{chr(10).join(example_blocks)}

Now answer the following question.

Question:
{row['question']}

Options:
{format_options(row)}
""".strip()

    return [
        {"role": "system", "content": "You answer industrial multiple-choice questions."},
        {"role": "user", "content": user_text},
    ]


def training_messages(row):
    """Prompt used for supervised fine-tuning."""
    user_text = f"""
Answer the following industrial failure-sensor multiple-choice question.
Return only one option label.

Question:
{row['question']}

Options:
{format_options(row)}
""".strip()

    return [
        {"role": "system", "content": "You answer industrial multiple-choice questions."},
        {"role": "user", "content": user_text},
    ]

In [ ]:
# =========================
# 7. Answer extraction
# =========================

def clean_label(label):
    """Remove punctuation from labels such as P) -> P."""
    return re.sub(r"[^A-Za-z0-9]", "", str(label)).upper()


def extract_answer(text, option_ids):
    """Extract a predicted option label from model output."""
    text = str(text).strip()
    text_upper = text.upper()

    # Try exact labels first.
    for label in option_ids:
        if str(label).upper() in text_upper:
            return label

    # Try cleaned labels.
    cleaned_to_original = {clean_label(label): label for label in option_ids}
    tokens = re.findall(r"[A-Za-z0-9]+", text_upper)
    for tok in tokens:
        if tok in cleaned_to_original:
            return cleaned_to_original[tok]

    # No valid label found.
    return None

In [ ]:
# =========================
# 8. Load Qwen2.5-7B-Instruct
# =========================

# No token or login is used here.
# If the environment has internet access, the model is downloaded from Hugging Face.
# On Kaggle, enable GPU before running this cell.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()
print("Model loaded.")

In [ ]:
# =========================
# 9. Text generation
# =========================

def generate_for_messages(messages_list, batch_size=GEN_BATCH_SIZE):
    """Generate answers for a list of chat messages."""
    outputs = []

    for start in range(0, len(messages_list), batch_size):
        batch_messages = messages_list[start:start + batch_size]

        prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in batch_messages
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only newly generated tokens.
        for i in range(len(prompts)):
            input_len = inputs["input_ids"].shape[1]
            new_tokens = generated[i, input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            outputs.append(text)

    return outputs

In [ ]:
# =========================
# 10. Evaluate LLM on one dataset
# =========================

def evaluate_llm_dataset(df, method_name, dataset_name, prompt_type, few_examples=None):
    """Evaluate model on a dataframe."""

    messages_list = []
    for _, row in df.iterrows():
        if prompt_type == "few_shot":
            messages = few_shot_messages(row, few_examples)
        else:
            messages = zero_shot_messages(row)
        messages_list.append(messages)

    start_time = time.time()
    generated_texts = generate_for_messages(messages_list)
    elapsed = time.time() - start_time

    rows = []
    for (_, row), gen_text in zip(df.iterrows(), generated_texts):
        pred_label = extract_answer(gen_text, row["option_ids"])
        invalid = pred_label is None
        correct = (pred_label == row["correct_label"])

        rows.append({
            "dataset": dataset_name,
            "method": method_name,
            "id": row["id"],
            "asset_name": row.get("asset_name", "unknown"),
            "question_type": row.get("question_type", "unknown"),
            "relevancy": row.get("relevancy", "unknown"),
            "n_options": row["n_options"],
            "correct_label": row["correct_label"],
            "pred_label": pred_label,
            "generated_text": gen_text,
            "correct": correct,
            "invalid": invalid,
        })

    result_df = pd.DataFrame(rows)
    result_df["time_per_question_sec"] = elapsed / max(len(df), 1)
    return result_df

In [ ]:
# =========================
# 11. Zero-shot and few-shot evaluation
# =========================

all_results = []

# Few-shot examples are selected from the training set.
few_examples = train_df.sample(
    n=min(N_FEW_SHOT_EXAMPLES, len(train_df)),
    random_state=RANDOM_SEED,
).to_dict("records")

if RUN_ZERO_SHOT:
    print("Running zero-shot evaluation...")
    all_results.append(evaluate_llm_dataset(test_df, "Zero-shot Qwen2.5-7B", "Original_Test", "zero_shot"))
    all_results.append(evaluate_llm_dataset(simple_test_df, "Zero-shot Qwen2.5-7B", "SimplePert", "zero_shot"))
    all_results.append(evaluate_llm_dataset(complex_test_df, "Zero-shot Qwen2.5-7B", "ComplexPert", "zero_shot"))

if RUN_FEW_SHOT:
    print("Running few-shot evaluation...")
    all_results.append(evaluate_llm_dataset(test_df, "Few-shot Qwen2.5-7B", "Original_Test", "few_shot", few_examples))
    all_results.append(evaluate_llm_dataset(simple_test_df, "Few-shot Qwen2.5-7B", "SimplePert", "few_shot", few_examples))
    all_results.append(evaluate_llm_dataset(complex_test_df, "Few-shot Qwen2.5-7B", "ComplexPert", "few_shot", few_examples))

print("Prompting evaluation finished.")

In [ ]:
# =========================
# 12. Prepare QLoRA training dataset
# =========================

class ChatMCQDataset(torch.utils.data.Dataset):
    """Dataset for supervised fine-tuning."""

    def __init__(self, df, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Prompt part.
        messages = training_messages(row)
        prompt_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # Answer part.
        answer_text = f" {row['correct_label']}{self.tokenizer.eos_token}"
        full_text = prompt_text + answer_text

        # Tokenize full text.
        full = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )

        # Tokenize prompt only.
        prompt = self.tokenizer(
            prompt_text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )

        input_ids = torch.tensor(full["input_ids"], dtype=torch.long)
        attention_mask = torch.tensor(full["attention_mask"], dtype=torch.long)

        # Labels are same as input ids, but prompt tokens are ignored with -100.
        labels = input_ids.clone()
        prompt_len = len(prompt["input_ids"])
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


class SimpleDataCollator:
    """Pads samples in a batch."""

    def __init__(self, tokenizer, label_pad_token_id=-100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            pad_len = max_len - len(f["input_ids"])

            input_ids.append(torch.cat([
                f["input_ids"],
                torch.full((pad_len,), self.tokenizer.pad_token_id, dtype=torch.long),
            ]))

            attention_mask.append(torch.cat([
                f["attention_mask"],
                torch.zeros(pad_len, dtype=torch.long),
            ]))

            labels.append(torch.cat([
                f["labels"],
                torch.full((pad_len,), self.label_pad_token_id, dtype=torch.long),
            ]))

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attention_mask),
            "labels": torch.stack(labels),
        }

In [ ]:
# =========================
# 13. QLoRA fine-tuning
# =========================

if RUN_QLORA:
    print("Preparing model for QLoRA training...")

    # Use right padding during training.
    tokenizer.padding_side = "right"

    model.train()
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    train_dataset = ChatMCQDataset(train_df, tokenizer, max_length=MAX_SEQ_LENGTH)
    data_collator = SimpleDataCollator(tokenizer)

    training_args = TrainingArguments(
        output_dir=str(MODEL_OUTPUT_DIR),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=MAX_GRAD_NORM,
        logging_steps=25,
        save_strategy="epoch",
        report_to="none",
        fp16=True,
        optim="paged_adamw_8bit",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
    )

    trainer.train()

    # Save only LoRA adapter and tokenizer.
    model.save_pretrained(str(MODEL_OUTPUT_DIR))
    tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

    print("QLoRA training finished.")

In [ ]:
# =========================
# 14. Evaluate QLoRA model
# =========================

if RUN_QLORA:
    # Use left padding again during generation.
    tokenizer.padding_side = "left"
    model.eval()

    print("Running QLoRA evaluation...")
    all_results.append(evaluate_llm_dataset(test_df, "QLoRA Qwen2.5-7B", "Original_Test", "zero_shot"))
    all_results.append(evaluate_llm_dataset(simple_test_df, "QLoRA Qwen2.5-7B", "SimplePert", "zero_shot"))
    all_results.append(evaluate_llm_dataset(complex_test_df, "QLoRA Qwen2.5-7B", "ComplexPert", "zero_shot"))

print("All LLM evaluations finished.")

In [ ]:
# =========================
# 15. Metrics tables
# =========================

llm_results = pd.concat(all_results, ignore_index=True)


def summary_table(result_df, dataset_name):
    """Create accuracy table for one dataset."""
    df = result_df[result_df["dataset"] == dataset_name]
    rows = []

    for method, group in df.groupby("method"):
        n = len(group)
        correct = int(group["correct"].sum())
        wrong = n - correct
        invalid = int(group["invalid"].sum())
        acc = 100 * correct / n
        invalid_rate = 100 * invalid / n
        t = float(group["time_per_question_sec"].iloc[0])

        rows.append({
            "Method": method,
            "N": n,
            "Accuracy (%)": round(acc, 2),
            "Correct": correct,
            "Wrong": wrong,
            "Invalid rate (%)": round(invalid_rate, 2),
            "Inference time / question (s)": round(t, 4),
        })

    return pd.DataFrame(rows).sort_values("Accuracy (%)", ascending=False)

main_summary = summary_table(llm_results, "Original_Test")
main_summary

In [ ]:
# =========================
# 16. Error analysis
# =========================

main_llm_results = llm_results[llm_results["dataset"] == "Original_Test"].copy()

error_by_question_type = (
    main_llm_results
    .groupby(["method", "question_type"])["correct"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

error_by_options = (
    main_llm_results
    .groupby(["method", "n_options"])["correct"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

error_by_asset = (
    main_llm_results
    .groupby(["asset_name", "method"])["correct"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

print("Error by question type")
display(error_by_question_type)

print("Error by number of options")
display(error_by_options)

print("Error by asset")
display(error_by_asset)

In [ ]:
# =========================
# 17. Disagreement analysis
# =========================

def disagreement(result_df, method_a, method_b):
    """Compare two methods on the same questions."""
    a = result_df[result_df["method"] == method_a][["id", "correct"]].rename(columns={"correct": "a_correct"})
    b = result_df[result_df["method"] == method_b][["id", "correct"]].rename(columns={"correct": "b_correct"})
    merged = a.merge(b, on="id")

    return {
        "Comparison": f"{method_a} vs {method_b}",
        "Both correct": int(((merged.a_correct == True) & (merged.b_correct == True)).sum()),
        "Both wrong": int(((merged.a_correct == False) & (merged.b_correct == False)).sum()),
        "First correct, second wrong": int(((merged.a_correct == True) & (merged.b_correct == False)).sum()),
        "First wrong, second correct": int(((merged.a_correct == False) & (merged.b_correct == True)).sum()),
    }

comparisons = []
methods_available = set(main_llm_results["method"].unique())

if {"Few-shot Qwen2.5-7B", "QLoRA Qwen2.5-7B"}.issubset(methods_available):
    comparisons.append(("Few-shot Qwen2.5-7B", "QLoRA Qwen2.5-7B"))
if {"Few-shot Qwen2.5-7B", "Zero-shot Qwen2.5-7B"}.issubset(methods_available):
    comparisons.append(("Few-shot Qwen2.5-7B", "Zero-shot Qwen2.5-7B"))
if {"QLoRA Qwen2.5-7B", "Zero-shot Qwen2.5-7B"}.issubset(methods_available):
    comparisons.append(("QLoRA Qwen2.5-7B", "Zero-shot Qwen2.5-7B"))

disagreement_table = pd.DataFrame([disagreement(main_llm_results, a, b) for a, b in comparisons])
disagreement_table

In [ ]:
# =========================
# 18. Robustness table
# =========================

def accuracy_by_method(dataset_name, column_name):
    df = llm_results[llm_results["dataset"] == dataset_name]
    return (
        df.groupby("method")["correct"]
        .mean()
        .mul(100)
        .round(2)
        .rename(column_name)
    )

orig_acc = accuracy_by_method("Original_Test", "Acc@Original (%)")
simple_acc = accuracy_by_method("SimplePert", "Acc@SimplePert (%)")
complex_acc = accuracy_by_method("ComplexPert", "Acc@ComplexPert (%)")

consistency_rows = []
for method in llm_results["method"].unique():
    orig = llm_results[(llm_results["dataset"] == "Original_Test") & (llm_results["method"] == method)][["id", "correct"]].rename(columns={"correct": "orig_correct"})
    comp = llm_results[(llm_results["dataset"] == "ComplexPert") & (llm_results["method"] == method)][["id", "correct"]].rename(columns={"correct": "complex_correct"})
    merged = orig.merge(comp, on="id")
    consist = 100 * ((merged["orig_correct"] == True) & (merged["complex_correct"] == True)).mean()
    consistency_rows.append({"method": method, "Acc@Consist-Complex (%)": round(consist, 2)})

consistency = pd.DataFrame(consistency_rows).set_index("method")
robustness_table = pd.concat([orig_acc, simple_acc, complex_acc, consistency], axis=1).reset_index()
robustness_table = robustness_table.rename(columns={"method": "Method"})
robustness_table

In [ ]:
# =========================
# 19. Save all results
# =========================

main_summary.to_csv(RESULTS_DIR / "llm_main_summary.csv", index=False)
llm_results.to_csv(RESULTS_DIR / "llm_all_predictions.csv", index=False)
robustness_table.to_csv(RESULTS_DIR / "llm_robustness_summary.csv", index=False)
disagreement_table.to_csv(RESULTS_DIR / "llm_disagreement.csv", index=False)

with pd.ExcelWriter(RESULTS_DIR / "llm_error_analysis.xlsx") as writer:
    main_summary.to_excel(writer, sheet_name="main_summary", index=False)
    error_by_question_type.to_excel(writer, sheet_name="by_question_type", index=False)
    error_by_options.to_excel(writer, sheet_name="by_num_options", index=False)
    error_by_asset.to_excel(writer, sheet_name="by_asset", index=False)
    disagreement_table.to_excel(writer, sheet_name="disagreement", index=False)
    robustness_table.to_excel(writer, sheet_name="robustness", index=False)

print("Saved result files to", RESULTS_DIR.resolve())